<a href="https://colab.research.google.com/github/acorrea79/techchallenge-fase2-pipeline-alfabetizacao/blob/main/notebooks/pipeline_alfabetizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TechChallenge Fase 2 - Pipeline Híbrido para Análise da Alfabetização no Brasil

Projeto desenvolvido para a FIAP, turma 1IAST.

Integrante:

- Andre Correa Luis Vilas Boas

## Objetivo do notebook

Este notebook centraliza a execução acadêmica da pipeline de dados para análise da alfabetização no Brasil.

A solução utiliza:

- GCP BigQuery Sandbox;
- Google Colab;
- Python;
- Pandas;
- arquitetura Medalhão com camadas Bronze, Silver e Gold;
- ingestão Batch;
- simulação de Streaming;
- validações de qualidade;
- logs de monitoramento.

Este notebook faz parte da entrega do TechChallenge Fase 2.

## Instalação das bibliotecas

In [ ]:
!pip install -q pandas numpy pyarrow google-cloud-bigquery db-dtypes

## Imports básicos do projeto

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

from google.colab import auth
from google.cloud import bigquery

## Configurações do projeto

In [ ]:
# Configurações principais do projeto

PROJECT_ID = "fiap-techchallenge-fase2"
DATASET_ID = "basedosdados.br_inep_avaliacao_alfabetizacao"

BASE_PATH = Path("/content/techchallenge-fase2-pipeline-alfabetizacao")

BRONZE_PATH = BASE_PATH / "data" / "bronze"
BRONZE_BATCH_PATH = BRONZE_PATH / "batch"
BRONZE_STREAMING_PATH = BRONZE_PATH / "streaming"

SILVER_PATH = BASE_PATH / "data" / "silver"
GOLD_PATH = BASE_PATH / "data" / "gold"
LOGS_PATH = BASE_PATH / "logs"

TABLES = {
    "alunos": f"{DATASET_ID}.alunos",
    "dicionario": f"{DATASET_ID}.dicionario",
    "meta_alfabetizacao_brasil": f"{DATASET_ID}.meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": f"{DATASET_ID}.meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": f"{DATASET_ID}.meta_alfabetizacao_municipio",
    "municipio": f"{DATASET_ID}.municipio",
    "uf": f"{DATASET_ID}.uf",
}

print("Configuração carregada com sucesso.")
print(f"Projeto GCP: {PROJECT_ID}")
print(f"Dataset principal: {DATASET_ID}")

## Criação das pastas da pipeline

In [ ]:
# Criação da estrutura de pastas no ambiente Colab

paths = [
    BRONZE_BATCH_PATH,
    BRONZE_STREAMING_PATH,
    SILVER_PATH,
    GOLD_PATH,
    LOGS_PATH,
]

for path in paths:
    path.mkdir(parents=True, exist_ok=True)

print("Estrutura de pastas criada no Colab:")
for path in paths:
    print(f"- {path}")

## Função simples de log

In [ ]:
# Função de log para monitoramento simples da pipeline

def write_log(message: str, level: str = "INFO"):
    LOGS_PATH.mkdir(parents=True, exist_ok=True)

    log_file = LOGS_PATH / "pipeline_execution.log"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    log_line = f"{timestamp} | {level} | {message}"

    with open(log_file, "a", encoding="utf-8") as file:
        file.write(log_line + "\n")

    print(log_line)


write_log("Notebook principal iniciado.")

## Autenticação no Google

In [ ]:
# Autenticação no Google
# Será aberta uma janela para autorizar o acesso com sua conta Google.

auth.authenticate_user()

write_log("Autenticação no Google realizada com sucesso.")

## Criação do cliente BigQuery

In [ ]:
# Criação do cliente BigQuery

client = bigquery.Client(project=PROJECT_ID)

write_log("Cliente BigQuery criado com sucesso.")

##Teste de acesso ao BigQuery

In [ ]:
# Teste simples de acesso ao BigQuery Sandbox

query = """
SELECT
  'BigQuery Sandbox acessado com sucesso' AS status
"""

df_test = client.query(query).to_dataframe(create_bqstorage_client=False)

display(df_test)

write_log("Consulta de teste executada com sucesso.")

# Listar tabelas do dataset principal

In [ ]:
!gcloud auth list
!gcloud config set project fiap-techchallenge-fase2
!gcloud config get-value project

query_tables = """
SELECT
  table_catalog,
  table_schema,
  table_name,
  table_type
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLES`
ORDER BY
  table_name
"""

df_tables = client.query(query_tables).to_dataframe(create_bqstorage_client=False)

display(df_tables)

write_log(f"Listagem de tabelas executada. Total de tabelas encontradas: {len(df_tables)}")

## Listar colunas do dataset principal

In [ ]:
# Listagem das colunas das tabelas do dataset principal

query_columns = """
SELECT
  table_name,
  column_name,
  data_type,
  is_nullable,
  ordinal_position
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.COLUMNS`
ORDER BY
  table_name,
  ordinal_position
"""

df_columns = client.query(query_columns).to_dataframe(create_bqstorage_client=False)

display(df_columns)

write_log(f"Schema carregado com sucesso. Total de colunas encontradas: {len(df_columns)}")

## Volume das tabelas

In [ ]:
# Consulta de volume das tabelas
# Esta etapa apoia decisões de FinOps.
# Tentamos primeiro INFORMATION_SCHEMA.TABLE_STORAGE.
# Caso falhe por restrição de localização/metadados, usamos __TABLES__ como alternativa.

query_storage_information_schema = """
SELECT
  table_name,
  total_rows,
  ROUND(total_logical_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLE_STORAGE`
ORDER BY
  total_logical_bytes DESC
"""

query_storage_legacy = """
SELECT
  table_id AS table_name,
  row_count AS total_rows,
  ROUND(size_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.__TABLES__`
ORDER BY
  size_bytes DESC
"""

try:
    df_storage = client.query(query_storage_information_schema).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "INFORMATION_SCHEMA.TABLE_STORAGE"
    write_log("Consulta de volume executada via INFORMATION_SCHEMA.TABLE_STORAGE.")

except Exception as error:
    write_log(
        f"Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. "
        f"Usando alternativa __TABLES__. Erro: {str(error)[:200]}",
        level="WARNING"
    )

    df_storage = client.query(query_storage_legacy).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "__TABLES__"
    write_log("Consulta alternativa de volume executada via __TABLES__.")

display(df_storage)

print(f"Fonte usada para volume das tabelas: {storage_source}")

## Validação das tabelas obrigatórias

In [ ]:
# Validação das tabelas obrigatórias do desafio

expected_tables = {
    "alunos",
    "dicionario",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
}

found_tables = set(df_tables["table_name"].tolist())

missing_tables = expected_tables - found_tables
extra_tables = found_tables - expected_tables

validation_result = {
    "expected_tables": sorted(list(expected_tables)),
    "found_tables": sorted(list(found_tables)),
    "missing_tables": sorted(list(missing_tables)),
    "extra_tables": sorted(list(extra_tables)),
    "status": "approved" if not missing_tables else "failed"
}

display(pd.DataFrame([validation_result]))

if missing_tables:
    write_log(f"Tabelas obrigatórias ausentes: {missing_tables}", level="ERROR")
else:
    write_log("Todas as tabelas obrigatórias foram encontradas.")

## Salvar metadados no ambiente do Colab

In [ ]:
# Salvando metadados de descoberta em arquivos locais do Colab

metadata_path = BRONZE_BATCH_PATH / "metadata"
metadata_path.mkdir(parents=True, exist_ok=True)

df_tables.to_csv(metadata_path / "dataset_tables.csv", index=False)
df_columns.to_csv(metadata_path / "dataset_columns.csv", index=False)
df_storage.to_csv(metadata_path / "dataset_storage.csv", index=False)

with open(metadata_path / "table_validation.json", "w", encoding="utf-8") as file:
    json.dump(validation_result, file, ensure_ascii=False, indent=4)

write_log("Metadados do dataset salvos na camada Bronze/metadata.")

## Verificar arquivos gerados

In [ ]:
# Verificação dos arquivos gerados

for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(str(BASE_PATH), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

## Nesta etapa, realizamos a ingestão batch dos dados públicos da Base dos Dados, utilizando o BigQuery Sandbox como componente cloud.

### A camada Bronze recebe os dados em formato bruto ou minimamente controlado, preservando a origem e permitindo rastreabilidade.

### Estratégia adotada:

- ingestão integral das tabelas pequenas;
- ingestão controlada da tabela `alunos`, devido ao maior volume;
- geração de uma amostra de alunos;
- geração de uma visão agregada de alunos por ano, município, rede e série;
- registro de manifesto da ingestão;
- registro de logs de execução.

### Essa abordagem segue a restrição acadêmica de custo zero e evita consumo desnecessário de processamento.

## Configuração da ingestão Batch

In [ ]:
# Configuração da ingestão Batch

BATCH_OUTPUT_PATH = BRONZE_BATCH_PATH

SMALL_TABLES = [
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
    "dicionario",
]

ALUNOS_SAMPLE_SIZE = 100000

batch_manifest = []

write_log(f"Tabelas pequenas configuradas para ingestão integral: {SMALL_TABLES}")
write_log(f"Tamanho da amostra da tabela alunos: {ALUNOS_SAMPLE_SIZE}")

## Função auxiliar para executar query e salvar CSV

In [ ]:
# Função auxiliar para executar consultas BigQuery e salvar resultado em CSV

def run_query_to_csv(query: str, output_file: Path, description: str):
    """
    Executa uma consulta BigQuery, converte o resultado para DataFrame
    e salva em CSV na camada Bronze.
    """

    start_time = time.perf_counter()
    started_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    write_log(f"Iniciando extração: {description}")

    try:
        job = client.query(query)
        df = job.to_dataframe(create_bqstorage_client=False)

        output_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_file, index=False, encoding="utf-8")

        elapsed_seconds = round(time.perf_counter() - start_time, 2)
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        file_size_mb = round(output_file.stat().st_size / 1024 / 1024, 4)

        manifest_item = {
            "description": description,
            "output_file": str(output_file),
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "bytes_processed": int(job.total_bytes_processed or 0),
            "file_size_mb": file_size_mb,
            "started_at": started_at,
            "finished_at": finished_at,
            "elapsed_seconds": elapsed_seconds,
            "status": "success"
        }

        batch_manifest.append(manifest_item)

        write_log(
            f"Extração concluída: {description} | "
            f"linhas={len(df)} | colunas={len(df.columns)} | "
            f"arquivo={output_file.name} | tamanho_mb={file_size_mb}"
        )

        return df

    except Exception as error:
        elapsed_seconds = round(time.perf_counter() - start_time, 2)
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        manifest_item = {
            "description": description,
            "output_file": str(output_file),
            "rows": 0,
            "columns": 0,
            "bytes_processed": 0,
            "file_size_mb": 0,
            "started_at": started_at,
            "finished_at": finished_at,
            "elapsed_seconds": elapsed_seconds,
            "status": "failed",
            "error": str(error)
        }

        batch_manifest.append(manifest_item)

        write_log(f"Erro na extração: {description} | erro={error}", level="ERROR")
        raise

## Ingestão integral das tabelas pequenas

In [ ]:
# Ingestão integral das tabelas pequenas para a camada Bronze

bronze_dataframes = {}

for table_name in SMALL_TABLES:
    table_id = TABLES[table_name]

    query = f"""
    SELECT *
    FROM `{table_id}`
    """

    output_file = BATCH_OUTPUT_PATH / f"{table_name}.csv"

    df = run_query_to_csv(
        query=query,
        output_file=output_file,
        description=f"Ingestão integral da tabela {table_name}"
    )

    bronze_dataframes[table_name] = df

write_log("Ingestão integral das tabelas pequenas concluída.")

## Ingestão controlada da tabela alunos: amostra

In [ ]:
# Ingestão controlada da tabela alunos
# A tabela alunos possui maior volume. Para custo zero, usamos amostra controlada.

query_alunos_sample = f"""
SELECT
  ano,
  id_municipio,
  id_escola,
  id_aluno,
  caderno,
  serie,
  rede,
  presenca,
  preenchimento_caderno,
  alfabetizado,
  proficiencia,
  peso_aluno
FROM
  `{TABLES["alunos"]}`
LIMIT {ALUNOS_SAMPLE_SIZE}
"""

alunos_sample_file = BATCH_OUTPUT_PATH / "alunos_sample.csv"

df_alunos_sample = run_query_to_csv(
    query=query_alunos_sample,
    output_file=alunos_sample_file,
    description=f"Ingestão controlada da tabela alunos com LIMIT {ALUNOS_SAMPLE_SIZE}"
)

display(df_alunos_sample.head())

write_log("Amostra da tabela alunos gerada com sucesso.")

## Ingestão agregada da tabela alunos

In [ ]:
# Ingestão agregada da tabela alunos
# Esta visão reduz o volume e permite análises por município, ano, rede e série.

query_alunos_agregado = f"""
SELECT
  ano,
  id_municipio,
  rede,
  serie,
  COUNT(*) AS total_alunos,
  COUNTIF(proficiencia IS NOT NULL) AS total_com_proficiencia,
  AVG(proficiencia) AS media_proficiencia,
  MIN(proficiencia) AS menor_proficiencia,
  MAX(proficiencia) AS maior_proficiencia,
  COUNTIF(alfabetizado IS NOT NULL) AS total_com_status_alfabetizacao,
  COUNT(DISTINCT alfabetizado) AS qtd_status_alfabetizacao,
  COUNTIF(presenca IS NOT NULL) AS total_com_status_presenca,
  AVG(peso_aluno) AS media_peso_aluno
FROM
  `{TABLES["alunos"]}`
GROUP BY
  ano,
  id_municipio,
  rede,
  serie
ORDER BY
  ano,
  id_municipio,
  rede,
  serie
"""

alunos_agregado_file = BATCH_OUTPUT_PATH / "alunos_agregado.csv"

df_alunos_agregado = run_query_to_csv(
    query=query_alunos_agregado,
    output_file=alunos_agregado_file,
    description="Ingestão agregada da tabela alunos por ano, município, rede e série"
)

display(df_alunos_agregado.head())

write_log("Visão agregada da tabela alunos gerada com sucesso.")

## Gerar manifesto da ingestão Batch

In [ ]:
# Geração do manifesto da ingestão Batch

manifest_file = BATCH_OUTPUT_PATH / "batch_ingestion_manifest.json"

with open(manifest_file, "w", encoding="utf-8") as file:
    json.dump(batch_manifest, file, ensure_ascii=False, indent=4)

write_log(f"Manifesto da ingestão Batch gerado: {manifest_file}")

df_manifest = pd.DataFrame(batch_manifest)
display(df_manifest)

## Validação dos arquivos gerados na Bronze

In [ ]:
# Validação dos arquivos gerados na camada Bronze Batch

expected_bronze_files = [
    "meta_alfabetizacao_brasil.csv",
    "meta_alfabetizacao_uf.csv",
    "meta_alfabetizacao_municipio.csv",
    "municipio.csv",
    "uf.csv",
    "dicionario.csv",
    "alunos_sample.csv",
    "alunos_agregado.csv",
    "batch_ingestion_manifest.json",
]

bronze_validation = []

for file_name in expected_bronze_files:
    file_path = BATCH_OUTPUT_PATH / file_name

    bronze_validation.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_bronze_validation = pd.DataFrame(bronze_validation)

display(df_bronze_validation)

if df_bronze_validation["exists"].all():
    write_log("Validação da camada Bronze Batch aprovada: todos os arquivos esperados foram gerados.")
else:
    missing = df_bronze_validation[df_bronze_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes na Bronze Batch: {missing}", level="ERROR")

## Resumo da ingestão Batch

In [ ]:
# Resumo executivo da ingestão Batch

total_files = len(expected_bronze_files)
total_rows = int(df_manifest["rows"].sum())
total_size_mb = round(df_bronze_validation["size_mb"].sum(), 4)
total_bytes_processed = int(df_manifest["bytes_processed"].sum())

summary_batch = {
    "total_files_generated": total_files,
    "total_rows_extracted": total_rows,
    "total_size_mb": total_size_mb,
    "total_bytes_processed": total_bytes_processed,
    "status": "approved" if df_bronze_validation["exists"].all() else "failed"
}

display(pd.DataFrame([summary_batch]))

write_log(
    f"Resumo Batch: arquivos={total_files}, linhas={total_rows}, "
    f"tamanho_mb={total_size_mb}, bytes_processados={total_bytes_processed}, "
    f"status={summary_batch['status']}"
)

## Listar arquivos gerados

In [ ]:
# Listagem dos arquivos gerados no ambiente Colab

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch -maxdepth 3 -type f

## Nesta etapa, os dados da camada Bronze são tratados e padronizados para compor a camada Silver.

### A camada Silver aplica:

- padronização de nomes e tipos de colunas;
- normalização de códigos de município e UF;
- remoção de duplicidades;
- tratamento estrutural de valores nulos;
- validação de campos obrigatórios;
- validação de faixas esperadas para percentuais;
- geração de relatório de qualidade;
- salvamento em formato Parquet.

### O uso de Parquet nas camadas tratadas contribui para otimização de armazenamento e leitura, reforçando a estratégia de FinOps do projeto.

### Configuração da camada Silver

In [ ]:
# Configuração da camada Silver

SILVER_OUTPUT_PATH = SILVER_PATH
SILVER_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

silver_manifest = []
silver_quality_report = []

write_log("Passo 7 iniciado: transformação da camada Bronze para Silver.")

Funções auxiliares da Silver

In [ ]:
# Funções auxiliares para transformação e validação da camada Silver

def read_bronze_csv(file_name: str) -> pd.DataFrame:
    """
    Lê um arquivo CSV da camada Bronze Batch.
    Alguns campos são forçados como string para preservar chaves e códigos.
    """
    file_path = BATCH_OUTPUT_PATH / file_name

    dtype_map = {
        "id_municipio": "string",
        "id_escola": "string",
        "id_aluno": "string",
        "sigla_uf": "string",
        "rede": "string",
        "serie": "string",
        "caderno": "string",
        "presenca": "string",
        "preenchimento_caderno": "string",
        "alfabetizado": "string",
        "id_tabela": "string",
        "nome_coluna": "string",
        "chave": "string",
        "cobertura_temporal": "string",
        "valor": "string",
    }

    df = pd.read_csv(file_path, dtype=dtype_map, low_memory=False)
    write_log(f"Arquivo Bronze lido: {file_name} | linhas={len(df)} | colunas={len(df.columns)}")

    return df


def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza nomes de colunas para minúsculas e sem espaços.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df


def normalize_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove espaços extras em colunas textuais.
    """
    df = df.copy()

    for column in df.columns:
        if pd.api.types.is_string_dtype(df[column]) or df[column].dtype == "object":
            df[column] = df[column].astype("string").str.strip()

    return df


def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza chaves principais como id_municipio e sigla_uf.
    """
    df = df.copy()

    if "id_municipio" in df.columns:
        df["id_municipio"] = (
            df["id_municipio"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(7)
        )

    if "sigla_uf" in df.columns:
        df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()

    if "rede" in df.columns:
        df["rede"] = df["rede"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

    if "serie" in df.columns:
        df["serie"] = df["serie"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

    return df


def convert_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte colunas numéricas conhecidas para os tipos adequados.
    """
    df = df.copy()

    integer_columns = [
        "ano",
        "nivel_alfabetizacao",
        "total_alunos",
        "total_com_proficiencia",
        "total_com_status_alfabetizacao",
        "qtd_status_alfabetizacao",
        "total_com_status_presenca",
    ]

    float_prefixes = [
        "taxa_",
        "meta_",
        "percentual_",
        "media_",
        "proporcao_",
        "proficiencia",
        "peso_",
        "menor_",
        "maior_",
    ]

    for column in integer_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")

    for column in df.columns:
        if any(column.startswith(prefix) for prefix in float_prefixes):
            df[column] = pd.to_numeric(df[column], errors="coerce")

    return df


def add_processing_metadata(df: pd.DataFrame, source_table: str) -> pd.DataFrame:
    """
    Adiciona metadados técnicos da transformação.
    """
    df = df.copy()
    df["source_table"] = source_table
    df["processed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df["layer"] = "silver"
    return df


def remove_duplicates(df: pd.DataFrame, subset_keys: list | None = None) -> tuple[pd.DataFrame, int]:
    """
    Remove duplicidades. Quando subset_keys é informado, usa as chaves de negócio.
    """
    df = df.copy()
    rows_before = len(df)

    if subset_keys:
        available_keys = [col for col in subset_keys if col in df.columns]
        df = df.drop_duplicates(subset=available_keys)
    else:
        df = df.drop_duplicates()

    rows_after = len(df)
    duplicates_removed = rows_before - rows_after

    return df, duplicates_removed


def validate_required_fields(df: pd.DataFrame, table_name: str, required_fields: list) -> dict:
    """
    Valida nulos em campos obrigatórios.
    """
    result = {
        "table_name": table_name,
        "validation_type": "required_fields",
        "status": "approved",
        "errors": {}
    }

    for field in required_fields:
        if field in df.columns:
            null_count = int(df[field].isna().sum())
            if null_count > 0:
                result["errors"][field] = null_count

    if result["errors"]:
        result["status"] = "warning"

    return result


def validate_percentage_ranges(df: pd.DataFrame, table_name: str) -> dict:
    """
    Valida campos percentuais esperados entre 0 e 100.
    Não altera os dados; apenas registra possíveis inconsistências.
    """
    percentage_columns = [
        col for col in df.columns
        if col.startswith("taxa_")
        or col.startswith("meta_alfabetizacao_")
        or col.startswith("percentual_")
        or col.startswith("proporcao_")
    ]

    result = {
        "table_name": table_name,
        "validation_type": "percentage_range_0_100",
        "status": "approved",
        "errors": {}
    }

    for column in percentage_columns:
        invalid_count = int(((df[column] < 0) | (df[column] > 100)).sum(skipna=True))
        if invalid_count > 0:
            result["errors"][column] = invalid_count

    if result["errors"]:
        result["status"] = "warning"

    return result


def transform_to_silver(
    source_file: str,
    output_file: str,
    source_table: str,
    required_fields: list,
    duplicate_keys: list | None = None
) -> pd.DataFrame:
    """
    Executa o fluxo completo de transformação Bronze -> Silver.
    """
    start_time = time.perf_counter()
    started_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    write_log(f"Iniciando transformação Silver: {source_file}")

    df = read_bronze_csv(source_file)

    rows_before = len(df)
    columns_before = len(df.columns)

    df = normalize_column_names(df)
    df = normalize_string_columns(df)
    df = normalize_keys(df)
    df = convert_numeric_columns(df)

    df, duplicates_removed = remove_duplicates(df, duplicate_keys)

    required_validation = validate_required_fields(df, output_file, required_fields)
    percentage_validation = validate_percentage_ranges(df, output_file)

    silver_quality_report.append(required_validation)
    silver_quality_report.append(percentage_validation)

    df = add_processing_metadata(df, source_table)

    output_path = SILVER_OUTPUT_PATH / output_file
    df.to_parquet(output_path, index=False)

    elapsed_seconds = round(time.perf_counter() - start_time, 2)
    finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    file_size_mb = round(output_path.stat().st_size / 1024 / 1024, 4)

    manifest_item = {
        "source_file": source_file,
        "output_file": str(output_path),
        "source_table": source_table,
        "rows_before": int(rows_before),
        "rows_after": int(len(df)),
        "columns_before": int(columns_before),
        "columns_after": int(len(df.columns)),
        "duplicates_removed": int(duplicates_removed),
        "file_size_mb": file_size_mb,
        "started_at": started_at,
        "finished_at": finished_at,
        "elapsed_seconds": elapsed_seconds,
        "status": "success"
    }

    silver_manifest.append(manifest_item)

    write_log(
        f"Transformação Silver concluída: {output_file} | "
        f"linhas={len(df)} | duplicidades_removidas={duplicates_removed} | "
        f"tamanho_mb={file_size_mb}"
    )

    return df

### Transformar tabelas de metas

In [ ]:
# Transformação das tabelas de metas para Silver

df_silver_meta_brasil = transform_to_silver(
    source_file="meta_alfabetizacao_brasil.csv",
    output_file="silver_meta_alfabetizacao_brasil.parquet",
    source_table="meta_alfabetizacao_brasil",
    required_fields=["ano", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "rede"]
)

df_silver_meta_uf = transform_to_silver(
    source_file="meta_alfabetizacao_uf.csv",
    output_file="silver_meta_alfabetizacao_uf.parquet",
    source_table="meta_alfabetizacao_uf",
    required_fields=["ano", "sigla_uf", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "sigla_uf", "rede"]
)

df_silver_meta_municipio = transform_to_silver(
    source_file="meta_alfabetizacao_municipio.csv",
    output_file="silver_meta_alfabetizacao_municipio.parquet",
    source_table="meta_alfabetizacao_municipio",
    required_fields=["ano", "id_municipio", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "id_municipio", "rede"]
)

display(df_silver_meta_brasil.head())
display(df_silver_meta_uf.head())
display(df_silver_meta_municipio.head())

### Transformar indicadores por município e UF

In [ ]:
# Transformação das tabelas de indicadores por município e UF para Silver

df_silver_indicador_municipio = transform_to_silver(
    source_file="municipio.csv",
    output_file="silver_indicador_municipio.parquet",
    source_table="municipio",
    required_fields=["ano", "id_municipio", "serie", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "id_municipio", "serie", "rede"]
)

df_silver_indicador_uf = transform_to_silver(
    source_file="uf.csv",
    output_file="silver_indicador_uf.parquet",
    source_table="uf",
    required_fields=["ano", "sigla_uf", "serie", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "sigla_uf", "serie", "rede"]
)

display(df_silver_indicador_municipio.head())
display(df_silver_indicador_uf.head())

### Transformar dados de alunos

In [ ]:
# Transformação das tabelas derivadas de alunos para Silver

df_silver_alunos_sample = transform_to_silver(
    source_file="alunos_sample.csv",
    output_file="silver_alunos_sample.parquet",
    source_table="alunos",
    required_fields=["ano", "id_municipio", "id_escola", "id_aluno", "serie", "rede"],
    duplicate_keys=["ano", "id_aluno"]
)

df_silver_alunos_agregado = transform_to_silver(
    source_file="alunos_agregado.csv",
    output_file="silver_alunos_agregado.parquet",
    source_table="alunos_agregado",
    required_fields=["ano", "id_municipio", "rede", "serie", "total_alunos"],
    duplicate_keys=["ano", "id_municipio", "rede", "serie"]
)

display(df_silver_alunos_sample.head())
display(df_silver_alunos_agregado.head())

### Transformar dicionário de dados

In [ ]:
# Transformação do dicionário de dados para Silver

df_silver_dicionario = transform_to_silver(
    source_file="dicionario.csv",
    output_file="silver_dicionario.parquet",
    source_table="dicionario",
    required_fields=["id_tabela", "nome_coluna", "chave", "valor"],
    duplicate_keys=["id_tabela", "nome_coluna", "chave"]
)

display(df_silver_dicionario.head(20))

### Salvar manifesto e relatório de qualidade da Silver

In [ ]:
# Salvando manifesto e relatório de qualidade da camada Silver

silver_manifest_file = SILVER_OUTPUT_PATH / "silver_transform_manifest.json"
silver_quality_file = SILVER_OUTPUT_PATH / "silver_quality_report.json"

with open(silver_manifest_file, "w", encoding="utf-8") as file:
    json.dump(silver_manifest, file, ensure_ascii=False, indent=4)

with open(silver_quality_file, "w", encoding="utf-8") as file:
    json.dump(silver_quality_report, file, ensure_ascii=False, indent=4)

df_silver_manifest = pd.DataFrame(silver_manifest)
df_silver_quality = pd.DataFrame(silver_quality_report)

display(df_silver_manifest)
display(df_silver_quality)

write_log(f"Manifesto Silver gerado: {silver_manifest_file}")
write_log(f"Relatório de qualidade Silver gerado: {silver_quality_file}")

### Validação dos arquivos Silver esperados

In [ ]:
# Validação dos arquivos esperados na camada Silver

expected_silver_files = [
    "silver_meta_alfabetizacao_brasil.parquet",
    "silver_meta_alfabetizacao_uf.parquet",
    "silver_meta_alfabetizacao_municipio.parquet",
    "silver_indicador_municipio.parquet",
    "silver_indicador_uf.parquet",
    "silver_alunos_sample.parquet",
    "silver_alunos_agregado.parquet",
    "silver_dicionario.parquet",
    "silver_transform_manifest.json",
    "silver_quality_report.json",
]

silver_validation = []

for file_name in expected_silver_files:
    file_path = SILVER_OUTPUT_PATH / file_name

    silver_validation.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_silver_validation = pd.DataFrame(silver_validation)

display(df_silver_validation)

if df_silver_validation["exists"].all():
    write_log("Validação da camada Silver aprovada: todos os arquivos esperados foram gerados.")
else:
    missing = df_silver_validation[df_silver_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes na camada Silver: {missing}", level="ERROR")

### Resumo da camada Silver

In [ ]:
# Resumo executivo da camada Silver

total_silver_files = len(expected_silver_files)
total_silver_rows = int(df_silver_manifest["rows_after"].sum())
total_silver_size_mb = round(df_silver_validation["size_mb"].sum(), 4)

silver_summary = {
    "total_files_generated": total_silver_files,
    "total_rows_processed": total_silver_rows,
    "total_size_mb": total_silver_size_mb,
    "total_duplicates_removed": int(df_silver_manifest["duplicates_removed"].sum()),
    "status": "approved" if df_silver_validation["exists"].all() else "failed"
}

display(pd.DataFrame([silver_summary]))

write_log(
    f"Resumo Silver: arquivos={total_silver_files}, linhas_processadas={total_silver_rows}, "
    f"tamanho_mb={total_silver_size_mb}, duplicidades_removidas={silver_summary['total_duplicates_removed']}, "
    f"status={silver_summary['status']}"
)

### Listar arquivos gerados na Silver

In [ ]:
# Listagem dos arquivos gerados na camada Silver

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver -maxdepth 2 -type f